In [6]:
import tkinter as tk
from tkinter import messagebox, ttk
from PIL import Image, ImageTk
import psycopg2
from psycopg2 import sql
import os

items = {
    "Phone"          ,
    "Laptop"          ,
    "Battery"        ,
    "Charger"         ,
    "Cable"           ,
    "Motherboard"    ,
    "Small Appliance" ,
    "Other"           
}

item_types = list(items)

root = tk.Tk()
root.title("ECO Track")
root.geometry("600x400")

Img = Image.open("rec.png")
Img = Img.resize((150, 150))
pic = ImageTk.PhotoImage(Img)

img = Image.open("rec.png")
img = img.resize((90, 90))
photo = ImageTk.PhotoImage(img)

current_user = ""

def connect_db():
    conn = psycopg2.connect(
        host="localhost",
        database="ewaste_db",
        user="postgres",
        password="cos101",
        port="5433"
    )
    cursor = conn.cursor()
    return conn, cursor

def create_tables():
    conn, cursor = connect_db()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users(
            id SERIAL PRIMARY KEY,
            username VARCHAR(100) UNIQUE NOT NULL,
            password VARCHAR(100) NOT NULL,
            hint VARCHAR(200)
        )
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS waste_logs(
            id SERIAL PRIMARY KEY,
            username VARCHAR(100) NOT NULL,
            item VARCHAR(100) NOT NULL,
            quantity INTEGER NOT NULL
        )
    """)
    conn.commit()
    conn.close()

def save_user(username, password, hint=""):
    conn, cursor = connect_db()
    try:
        cursor.execute("""
            INSERT INTO users(username, password, hint)
            VALUES (%s, %s, %s)
        """, (username, password, hint))
        conn.commit()
        return True
    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        return False
    finally:
        conn.close()

def check_login(username, password):
    conn, cursor = connect_db()
    cursor.execute("""
        SELECT * FROM users
        WHERE username=%s AND password=%s
    """, (username, password))
    user = cursor.fetchone()
    conn.close()
    return user

def save_log(item, quantity):
    conn, cursor = connect_db()
    cursor.execute("""
        INSERT INTO waste_logs(username, item, quantity)
        VALUES (%s, %s, %s)
    """, (current_user, item, quantity))
    conn.commit()
    conn.close()


def clear_page():
    for widget in root.winfo_children():
        widget.destroy()

def footer():
    footer = tk.Label( text="EcoTrack © 2026",font=("Arial", 9),fg="gray")
    footer.place(x=280, y=370)

def build_header(subtitle=None):
    left_frame = tk.Frame(root, bg="green", width=70, height=400)
    left_frame.place(x=50, y=0)

    image_label = tk.Label(root,image=photo)
    image_label.place(x=135, y=30)

    tk.Label(root, text="ECO Track", fg="green",font=("Arial", 37, "bold")).place(x=230, y=30)

    if subtitle:
        tk.Label(root, text=subtitle,
                 font=("Arial", 18, "bold")).place(x=230, y=83)

    canvas = tk.Canvas(root, width=20, height=180)
    canvas.place(x=210, y=120)
    canvas.create_line(10, 10, 10, 170, fill="green", width=1)

def sign_up():
    clear_page()
    build_header()
    footer()
    tk.Label(root, text="Set Username:", font=("Arial", 10)).place(x=230, y=140)
    entry = tk.Entry(root, width=40)
    entry.place(x=234, y=160)

    tk.Label(root, text="Set Password:", font=("Arial", 10)).place(x=230, y=195)
    pass_entry = tk.Entry(root, width=40, show="*")
    pass_entry.place(x=234, y=215)

    tk.Label(root, text="Password Hint:", font=("Arial", 10)).place(x=230, y=250)
    hint_entry = tk.Entry(root, width=40)
    hint_entry.place(x=234, y=270)

    def signed_up():
        user = entry.get().strip()
        pw = pass_entry.get().strip()
        hint = hint_entry.get().strip()

        if user == "" or pw == "":
            messagebox.showwarning("Missing Info", "Username and Password cannot be empty")
            return

        success = save_user(user, pw, hint)
        if not success:
            messagebox.showwarning("Taken", "That username is already registered.")
            return

        messagebox.showinfo("Success", "Account created! Please log in.")
        login()

    tk.Button(root, text="Sign Up", bg="white", fg="green", width=34,
              command=signed_up).place(x=234, y=310)
    tk.Button(root, text="← Back to Login", font=("Arial", 9), fg="green",
              bg="white", bd=0, command=login).place(x=234, y=350)


def log_item():
    clear_page()
    build_header("Online Waste Tracker")
    footer()
    tk.Label(root, text="Welcome, " + current_user,
             font=("Arial", 12)).place(x=230, y=115)

    tk.Label(root, text="Item:", font=("Arial", 10)).place(x=230, y=140)
    item_menu = ttk.Combobox(textvariable=item_types,values=item_types, width=37)
    item_menu.place(x=234,y=170)

    tk.Label(root, text="Quantity:", font=("Arial", 10)).place(x=230, y=195)
    quantity_entry = tk.Entry(root, width=40)
    quantity_entry.place(x=234, y=215)

    def submit_item():
        item = item_menu.get().strip()
        qty = quantity_entry.get().strip()

        if item == "" or qty == "":
            messagebox.showwarning("Missing Info", "Item and Quantity cannot be empty")
            return
        try:
            qty_int = int(qty)
        except ValueError:
            messagebox.showerror("Invalid Input", "Quantity must be a whole number")
            return

        save_log(item, qty_int) 
        messagebox.showinfo("Saved", "Item logged successfully!")
        item_menu.delete(0, tk.END)
        quantity_entry.delete(0, tk.END)

    tk.Button(root, text="Log item", bg="green", fg="white", width=34,
              command=submit_item).place(x=234, y=250)
    tk.Button(root, text="Log Out", width=10, command=login).place(x=500, y=10)


def login():
    clear_page()
    build_header("Online Waste Tracker")
    footer()
    tk.Label(root, text="Username:", font=("Arial", 10)).place(x=230, y=140)
    username_entry = tk.Entry(root, width=40)
    username_entry.place(x=234, y=160)

    tk.Label(root, text="Password:", font=("Arial", 10)).place(x=230, y=195)
    password_entry = tk.Entry(root, width=40, show="*")
    password_entry.place(x=234, y=215)

    def try_login():
        global current_user
        user = username_entry.get().strip()
        pw = password_entry.get().strip()

        if user == "" or pw == "":
            messagebox.showwarning("Missing Info", "Username and Password cannot be empty")
            return

        result = check_login(user, pw)
        if not result:
            messagebox.showerror("Failed", "Invalid username or password.")
            return

        current_user = user
        log_item()

    tk.Button(root, text="LOGIN", bg="green", fg="white", width=14,
              command=try_login).place(x=234, y=250)
    tk.Button(root, text="SIGN UP", width=14, command=sign_up,
              bg="white", fg="green").place(x=370, y=250)



create_tables() 
login()
root.mainloop()